# 12.5 asyncio

**Prerequisites:** 12.1 (the GIL), 12.2 Threading, 11.4 (which used asyncio informally)  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- Coroutines, the event loop, and what `await` actually does
- 🔴 **A coroutine does nothing until you await it** - the commonest first mistake
- 🔴 Running `asyncio` inside Jupyter, where a loop is already going
- `gather()`, and **`TaskGroup`** (3.11+) which supersedes it
- Timeouts: `wait_for()` and **`asyncio.timeout()`** (3.11+)
- 🔴 One blocking call stalls everything - and `asyncio.to_thread()`
- `Semaphore` for rate limiting; cancellation; async context managers and iterators
- 🔴 Fire-and-forget tasks being garbage collected mid-flight
- Choosing between `asyncio` and threads

---

## The idea

**12.2** used threads to overlap waiting, and paid for it with locks and races. `asyncio` overlaps waiting on **one thread**, so most of that danger disappears.

```
   THREADS                            ASYNCIO
   the OS interrupts you anywhere     you yield only at `await`
   pre-emptive                        cooperative
   needs locks                        needs no locks between awaits
   ~50 µs and ~8 MB per thread*       ~1 µs and a few KB per task
   hundreds of them                   tens of thousands
```

\*The ~8 MB is the thread stack's **virtual reservation** — address space the OS commits
page by page as it is actually used, not resident RAM. The cost is still real (address space,
kernel bookkeeping, and the ~50 µs switches are paid in full), just not eight megabytes of
memory per thread.

### The restaurant

A thread-based restaurant hires one waiter per table. Each stands at their table doing nothing while the kitchen cooks.

An async restaurant has **one waiter** who takes an order, hands it to the kitchen, and immediately moves to the next table. They are never idle and never in two places at once. The meals arrive no faster — but one waiter serves the whole room.

🔴 And if that waiter sits down to peel potatoes themselves, **every table waits**. That is a blocking call, and it is the one way to ruin async code.

## Three words

| Word | Is |
|---|---|
| **coroutine** | what `async def` creates. Calling it returns an object; it does not run |
| **event loop** | the scheduler. Runs one coroutine until it awaits, then picks another |
| **task** | a coroutine handed to the loop to run *concurrently* |

```
    async def fetch(): ...        <- defines a coroutine function

    fetch()                       <- creates a coroutine OBJECT. Runs nothing.
    await fetch()                 <- runs it, and waits for it
    asyncio.create_task(fetch())  <- schedules it to run CONCURRENTLY
```

🔴 That second line is the classic first bug: calling an `async def` function without `await` does nothing at all, and Python only warns you at garbage-collection time.

### 🔴 Running asyncio in a notebook

Jupyter **already runs an event loop** to talk to the front end, so:

```
    asyncio.run(main())    -> RuntimeError: asyncio.run() cannot be called
                              from a running event loop
```

| Where the code runs | What works |
|---|---|
| A `.py` script | `asyncio.run(main())` |
| A Jupyter cell | bare `await main()` - Jupyter permits top-level `await` |
| **Both** | the `run_async` helper below |

The helper checks whether a loop is running and, if so, gives the coroutine its own loop on a separate thread. Every cell in this notebook uses it, so the notebook behaves identically in Jupyter and under a plain interpreter.

In [ ]:
import asyncio
import random
import time
from concurrent.futures import ThreadPoolExecutor


def run_async(coro):
    """Run a coroutine whether or not an event loop is already running."""
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)              # no loop: a plain script
    with ThreadPoolExecutor(1) as pool:       # a loop IS running: Jupyter
        return pool.submit(asyncio.run, coro).result()


async def hello(name):
    await asyncio.sleep(0.1)                  # yields control to the loop
    return f"hello, {name}"


# 🔴 Calling it does NOT run it
coro = hello("world")
print("calling an async def gives:", type(coro).__name__)
print("has it run? no - nothing has happened yet")
coro.close()                                  # avoid 'never awaited' warning

print("\nawaited via run_async:", run_async(hello("world")))

## Concurrency needs *tasks*, not just `await`

This is the second classic mistake. `await` means *wait for this now*. Awaiting three coroutines one after another is **sequential** — you have written async code that behaves exactly like blocking code.

```
    await a(); await b(); await c()      SEQUENTIAL - 3 x the time

    await asyncio.gather(a(), b(), c())  CONCURRENT - as slow as the slowest
```

Concurrency starts when several coroutines are **in flight at once**, which means `gather()`, `create_task()`, or a `TaskGroup`.

In [ ]:
async def step(name, delay):
    await asyncio.sleep(delay)
    return f"{name} done"


async def sequential():
    started = time.perf_counter()
    results = [await step("a", 0.2), await step("b", 0.2), await step("c", 0.2)]
    return results, time.perf_counter() - started


async def concurrent():
    started = time.perf_counter()
    results = await asyncio.gather(
        step("a", 0.2), step("b", 0.2), step("c", 0.2)
    )
    return results, time.perf_counter() - started


results, seq_time = run_async(sequential())
print(f"awaited one by one : {seq_time:.2f}s  {results}")

results, con_time = run_async(concurrent())
print(f"gather()           : {con_time:.2f}s  {results}")

print(f"\n{seq_time / con_time:.1f}x - and the only change was gather().")
print("Both are 'async code'. Only one is concurrent.")

## `TaskGroup` - what to use instead of `gather()`

> **Version note — 3.11+.** `asyncio.TaskGroup` is the modern replacement for `gather()`, and it is better in the way that matters: **error handling**.

| | `gather()` | `TaskGroup` |
|---|---|---|
| One task fails | others **keep running**, unsupervised | all siblings are cancelled |
| Multiple failures | only the first is raised | **`ExceptionGroup`** with all of them |
| Structure | tasks can outlive the call | nothing escapes the block |

🔴 `gather()`'s default behaviour is a real hazard: a failure leaves the other tasks running with nobody waiting on them. `TaskGroup` gives **structured concurrency** — when the block exits, every task it started is finished or cancelled.

Catch its failures with `except*`, the 3.11 syntax for `ExceptionGroup`.

In [ ]:
async def unreliable(name, delay, fail=False):
    await asyncio.sleep(delay)
    if fail:
        raise ValueError(f"{name} failed")
    return name


async def with_gather():
    try:
        await asyncio.gather(
            unreliable("a", 0.05, fail=True),
            unreliable("b", 0.05, fail=True),
            unreliable("c", 0.30),
        )
    except ValueError as exc:
        return f"gather raised only: {exc}"


async def with_taskgroup():
    collected = []
    try:
        async with asyncio.TaskGroup() as group:
            group.create_task(unreliable("a", 0.05, fail=True))
            group.create_task(unreliable("b", 0.05, fail=True))
            group.create_task(unreliable("c", 0.30))
    except* ValueError as group_exc:                 # 3.11+ syntax
        collected = [str(e) for e in group_exc.exceptions]
    return collected


print(run_async(with_gather()))
print("  ^ the second failure is lost, and 'c' was abandoned - unsupervised")
print("    until asyncio.run's shutdown cancelled it\n")

print("TaskGroup reports every failure:", run_async(with_taskgroup()))
print("  ^ both errors, and 'c' was cancelled rather than abandoned")

## Timeouts

| | Since | Shape |
|---|---|---|
| `asyncio.wait_for(coro, timeout)` | always | wraps one awaitable |
| `async with asyncio.timeout(t):` | **3.11+** | wraps a whole **block** |

`asyncio.timeout()` is usually clearer, because a real operation is several awaits and you want a deadline over all of them rather than each one.

Both raise `TimeoutError` and **cancel** the work — unlike threads (**12.2**), where `join(timeout=)` leaves the thread running. Cancellation is one thing async genuinely does better.

In [ ]:
async def slow_operation():
    await asyncio.sleep(5)
    return "finished"


async def demo_timeouts():
    out = []

    started = time.perf_counter()
    try:
        await asyncio.wait_for(slow_operation(), timeout=0.2)
    except TimeoutError:
        out.append(f"wait_for      -> TimeoutError after "
                   f"{time.perf_counter() - started:.2f}s")

    started = time.perf_counter()
    try:
        async with asyncio.timeout(0.2):        # 3.11+, wraps a whole block
            await asyncio.sleep(0.05)
            await slow_operation()
    except TimeoutError:
        out.append(f"timeout block -> TimeoutError after "
                   f"{time.perf_counter() - started:.2f}s")

    return out


for line in run_async(demo_timeouts()):
    print("  " + line)
print("\n✅ The work was CANCELLED, not merely abandoned - unlike a thread,")
print("   which keeps running after join(timeout=) gives up (12.2).")

## 🔴 One blocking call stalls everything

The defining hazard, measured in **11.4** at a 5.8x slowdown.

There is **one thread**. A coroutine holds it until it awaits. Call anything that blocks — `time.sleep`, `requests.get`, a synchronous database driver, a big CPU loop — and every other task is frozen.

| Blocking | Async equivalent |
|---|---|
| `time.sleep(n)` | `await asyncio.sleep(n)` |
| `requests.get(url)` | `httpx` / `aiohttp` |
| `open(...).read()` | `aiofiles`, or `asyncio.to_thread` |
| any blocking call | **`await asyncio.to_thread(func, *args)`** |

`asyncio.to_thread()` (3.9+) runs a blocking function on a worker thread and awaits the result — the bridge between this notebook and **12.2**. For CPU-bound work use `run_in_executor` with a **ProcessPool** (**12.3**), since a thread would not help.

In [ ]:
def blocking_call(seconds):
    """A synchronous library you cannot change."""
    time.sleep(seconds)
    return seconds


async def all_blocking():
    started = time.perf_counter()
    results = []
    for _ in range(4):
        results.append(blocking_call(0.25))       # 🔴 blocks the whole loop
    return time.perf_counter() - started


async def properly_offloaded():
    started = time.perf_counter()
    await asyncio.gather(*(asyncio.to_thread(blocking_call, 0.25) for _ in range(4)))
    return time.perf_counter() - started


blocked = run_async(all_blocking())
offloaded = run_async(properly_offloaded())

print(f"blocking calls inside the loop : {blocked:.2f}s")
print(f"via asyncio.to_thread          : {offloaded:.2f}s")
print(f"\n{blocked / offloaded:.1f}x, and the code looks almost identical.")
print("\nNothing warns you about the first version. It is 'async code' that")
print("has quietly become sequential - the single most common asyncio bug.")

## Limiting concurrency with `Semaphore`

`gather()` over 10,000 URLs starts 10,000 requests at once, which will exhaust your file descriptors and get you rate-limited (**11.5**).

`asyncio.Semaphore(n)` caps how many run concurrently — the async twin of the threading `Semaphore` in **12.2**, and the standard shape for any real client.

In [ ]:
MAX_CONCURRENT = 3
in_flight = 0
peak = 0


async def limited_fetch(name, limiter):
    global in_flight, peak
    async with limiter:                 # waits here if 3 are already inside
        in_flight += 1                  # no lock needed: single thread,
        peak = max(peak, in_flight)     # and no await between these lines
        await asyncio.sleep(0.1)
        in_flight -= 1
    return name


async def fetch_many(count):
    limiter = asyncio.Semaphore(MAX_CONCURRENT)
    started = time.perf_counter()
    async with asyncio.TaskGroup() as group:
        for i in range(count):
            group.create_task(limited_fetch(f"url-{i}", limiter))
    return time.perf_counter() - started


elapsed = run_async(fetch_many(12))
print(f"12 tasks, Semaphore({MAX_CONCURRENT}) -> {elapsed:.2f}s, peak concurrent: {peak}")
print(f"   (12 / {MAX_CONCURRENT} batches x 0.1s = {12 / MAX_CONCURRENT * 0.1:.1f}s)")
print()
print("Note the counter needed NO lock. Between two awaits nothing else")
print("runs, so the read-modify-write cannot be interrupted - the race from")
print("12.2 is structurally impossible here.")

## 🔴 Fire-and-forget tasks can vanish

A subtle one that bites real services:

```
    asyncio.create_task(background_job())     <- nothing holds a reference
```

The event loop keeps only a **weak** reference to a task. If nothing else refers to it, it can be garbage-collected **mid-execution**, and the job simply stops — no error, no log.

**The fix:** keep a strong reference, and remove it when the task completes.

```
    _background = set()
    task = asyncio.create_task(job())
    _background.add(task)
    task.add_done_callback(_background.discard)
```

Better still, use a `TaskGroup`, which holds its tasks for you — another reason it supersedes loose `create_task` calls.

In [ ]:
completed = []


async def background_job(n):
    await asyncio.sleep(0.05)
    completed.append(n)


async def keeping_references():
    background = set()                        # the strong references
    for n in range(5):
        task = asyncio.create_task(background_job(n))
        background.add(task)
        task.add_done_callback(background.discard)
    await asyncio.sleep(0.2)                  # let them finish
    return len(background)


remaining = run_async(keeping_references())
print("jobs that completed :", sorted(completed))
print("references remaining:", remaining, "(the callback discarded each one)")

completed.clear()


async def with_taskgroup_instead():
    async with asyncio.TaskGroup() as group:  # holds them for you
        for n in range(5):
            group.create_task(background_job(n))
    return "all finished before the block exited"


print("\nTaskGroup:", run_async(with_taskgroup_instead()))
print("jobs completed:", sorted(completed))

## Async context managers and iterators

Two pieces of syntax you will meet constantly in async libraries:

```
    async with session.get(url) as response:    __aenter__ / __aexit__
    async for row in cursor:                    __aiter__ / __anext__
```

They exist because setup, teardown and iteration may themselves need to wait — opening a connection, fetching the next page of results. The synchronous `with` and `for` from **6.3** and **3.3** cannot await.

An **async generator** — `async def` containing `yield` — is the easy way to write an async iterator.

In [ ]:
class Connection:
    """An async context manager: setup and teardown can both await."""

    def __init__(self, name):
        self.name = name
        self.events = []

    async def __aenter__(self):
        await asyncio.sleep(0.02)             # a real connect() would wait
        self.events.append("opened")
        return self

    async def __aexit__(self, exc_type, exc, tb):
        await asyncio.sleep(0.02)             # so would a real close()
        self.events.append("closed")
        return False                          # do not suppress exceptions


async def stream_rows(count):
    """An async generator - the easy way to write an async iterator."""
    for i in range(count):
        await asyncio.sleep(0.01)             # e.g. fetching the next page
        yield {"id": i, "state": "queued" if i % 2 else "done"}


async def demo():
    async with Connection("jobs-db") as conn:
        rows = [row async for row in stream_rows(4)]      # async comprehension
    return conn.events, rows


events, rows = run_async(demo())
print("connection lifecycle:", events)
for row in rows:
    print("  ", row)
print("\n`async with` and `async for` exist because setup, teardown and")
print("fetching the next item can all need to wait.")

## Choosing: `asyncio` or threads?

| Use `asyncio` when | Use threads when |
|---|---|
| Hundreds or thousands of concurrent I/O operations | a few dozen |
| Async libraries exist for what you call | your libraries are synchronous |
| You are writing a server or client from scratch | you are adding concurrency to existing code |
| You want no locks | you want no rewrite |

### The honest summary

`asyncio` is **not faster than threads** for a handful of tasks — **11.4** measured them at 0.31s and 0.32s for six clients. It wins on **scale** (tens of thousands of tasks against hundreds of threads) and on **structure** (no locks, no races).

It costs you: a rewrite of everything on the call path, since one blocking call ruins it; a smaller ecosystem; and harder debugging.

> **The pragmatic rule.** Adding concurrency to existing synchronous code? Threads or `ThreadPoolExecutor` (**12.4**). Building something new that will hold many thousands of connections? `asyncio`. Doing CPU work? Neither — processes (**12.3**).

| Folder | Covered |
|---|---|
| **12.1** | concurrency vs parallelism, the GIL |
| **12.2** | threads, races, locks, queues |
| **12.3** | processes, real parallelism, the spawn guard |
| **12.4** | `concurrent.futures` over both |
| **12.5** | `asyncio` |

In [ ]:
import threading

leftover = [t.name for t in threading.enumerate() if t is not threading.main_thread()]
print("threads still alive:", leftover or "none")
print("\nEvery coroutine here ran inside run_async, which closes its loop")
print("when it returns - so no event loop is left running either.")

---

## Common Mistakes & Pitfalls

1. 🔴 **Calling an `async def` without awaiting it.** Nothing runs. You get a coroutine object and a warning at garbage-collection time, if you are lucky.
2. 🔴 **Awaiting coroutines one after another and calling it concurrent.** That is sequential. Use `gather()` or a `TaskGroup`.
3. 🔴 **Any blocking call inside a coroutine.** One `time.sleep` or `requests.get` freezes every task. Use `asyncio.to_thread()`.
4. 🔴 **`asyncio.run()` in a Jupyter cell.** A loop is already running - use bare `await`, or a helper.
5. 🔴 **Fire-and-forget `create_task()` with no reference kept.** The task can be garbage-collected mid-execution. Keep a set, or use a `TaskGroup`.
6. **`gather()` when a failure should stop the others.** It leaves siblings running unsupervised and reports only the first error. `TaskGroup` does both properly.
7. **Unbounded concurrency.** `gather()` over 10,000 items starts 10,000 operations. Use a `Semaphore`.
8. **Expecting CPU-bound work to benefit.** One thread, one GIL - `asyncio` does nothing for it. Use processes (**12.3**).
9. **Mixing `time.sleep` and `asyncio.sleep`.** The first blocks the loop; the second yields.

## Best Practices

- Prefer `TaskGroup` (3.11+) to `gather()` - structured concurrency, and every error reported.
- Prefer `async with asyncio.timeout(...)` to `wait_for` for multi-step operations.
- Bound concurrency with a `Semaphore` on anything that touches a remote service.
- Wrap unavoidable blocking calls in `asyncio.to_thread()`; use a process pool for CPU work.
- Keep strong references to background tasks, and clear them in a done-callback.
- Check that every library on the async path is non-blocking before you commit.
- Give tasks names (`create_task(coro, name=...)`) - it is what appears in debugging output.
- Turn on `asyncio` debug mode (`PYTHONASYNCIODEBUG=1`) while developing; it reports slow callbacks and un-awaited coroutines.

> **Version note — 3.14.** Inspection improved: `python -m asyncio ps <PID>` and
> `python -m asyncio pstree <PID>` list every pending task in a *running* process, await
> chains included — exactly the view you want when a production loop is stuck.

## Practice Exercises

Try these before moving on.

1. Take the sequential example and convert it three ways - `gather`, `TaskGroup`, and explicit `create_task` plus `await`. Which reads best?
2. 🔴 Make two tasks in a `TaskGroup` fail with different exception types and catch them with two `except*` clauses. What does `gather()` do with the same code?
3. Add a `Semaphore` to the `TaskGroup` example limiting concurrency to 2, and verify the peak by recording it.
4. Write `fetch_all(urls, limit)` that fetches with `asyncio.to_thread` plus `requests` (**11.5**), bounded by a `Semaphore`, with a per-request timeout.
5. Implement an async retry decorator with exponential backoff that gives up after N attempts and re-raises the last error.
6. Write an async generator that pages through results, yielding one row at a time while fetching the next page in the background.
7. 🔴 Create a task, drop every reference to it, force `gc.collect()`, and see whether it completes. Then keep a reference and compare.